In [51]:
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, END
import os
import json
import time
from dotenv import load_dotenv
from typing import Any, Dict, List, Optional, Literal, TypedDict, Annotated, operator
from dataclasses import dataclass, asdict
import re
from datetime import datetime, timezone

In [11]:
load_dotenv()
GOOGLE_API_KEY = os.getenv("GEMINI_API_KEY")

In [15]:

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview", 
    api_key=GOOGLE_API_KEY,
    temperature=0.1)

llm.invoke("hi").text

'Hello! How can I help you today?'

In [47]:
class DevState(TypedDict):
    raw_input:           str
    intent_manifest:     Optional[Dict]
    compliance_rules:    Optional[Dict]
    ip_clearance:        Optional[Dict]
    architecture:        Optional[Dict]
    generated_code:      Optional[Dict]
    explainability_docs: Optional[Dict]
    security_report:     Optional[Dict]
    quality_report:      Optional[Dict]
    hitl_decisions:      Optional[List]
    audit_log:           Annotated[List[Dict], operator.add]
    drift_alerts:        Optional[List]


In [53]:
def extract_json(text: str) -> dict:
    text = re.sub(r"```(?:json)?", "", text).replace("```", "").strip()
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON object found in LLM response:\n{text}")
    return json.loads(match.group())


def make_audit_entry(agent: str, summary: str, data: dict) -> dict:

    return {
        "agent":     agent,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "summary":   summary,
        "data":      data,
    }

In [54]:
from datetime import datetime, timezone

timestamp = datetime.now(timezone.utc).isoformat()

print(timestamp)

2026-03-13T12:40:32.921372+00:00


In [55]:
INTENT_SCHEMA = """
{
  "app_type": "string — e.g. REST API, CLI tool, web app",
  "modules": [
    {
      "name": "string",
      "description": "string",
      "tech_stack": ["string"]
    }
  ],
  "constraints": {
    "security": ["string"],
    "compliance": ["string — e.g. GDPR, HIPAA"],
    "performance": ["string"],
    "ip_notes": ["string — any third-party libraries or frameworks mentioned"]
  },
  "acceptance_criteria": ["string"]
}
"""

def intent_agent(state: DevState) -> dict:
    prompt = f"""
You are an AI software architect. Convert the user's description into a
structured intent manifest. Respond ONLY with valid JSON — no explanation,
no markdown, no extra text.

User input:
{state["raw_input"]}

Return exactly this JSON schema (fill in real values, do not include comments):
{INTENT_SCHEMA}
"""

    response = llm.invoke(prompt)
    manifest = extract_json(response.text)

    audit_entry = make_audit_entry(
        agent   = "intent_agent",
        summary = f"Parsed intent for: {state['raw_input'][:80]}",
        data    = {"app_type": manifest.get("app_type"), "modules": [m["name"] for m in manifest.get("modules", [])]}
    )

    return {
        "intent_manifest": manifest,
        "audit_log":       [audit_entry],   
    }


In [56]:
def main():
    state: DevState = {
        "raw_input":           "Build a FastAPI web app with login and dashboard using PostgreSQL",
        "intent_manifest":     None,
        "compliance_rules":    None,
        "ip_clearance":        None,
        "architecture":        None,
        "generated_code":      None,
        "explainability_docs": None,
        "security_report":     None,
        "quality_report":      None,
        "hitl_decisions":      None,
        "audit_log":           [],
        "drift_alerts":        None,
    }

    print("=== Running intent_agent ===")
    result = intent_agent(state)
    state["intent_manifest"] = result["intent_manifest"]
    state["audit_log"] += result["audit_log"]   # simulate operator.add

    print("\nIntent Manifest:")
    print(json.dumps(state["intent_manifest"], indent=2))

    print("\nAudit Log:")
    print(json.dumps(state["audit_log"], indent=2))

if __name__ == "__main__":
    main()

=== Running intent_agent ===

Intent Manifest:
{
  "app_type": "web app",
  "modules": [
    {
      "name": "Authentication Module",
      "description": "User registration and login system with JWT-based session management.",
      "tech_stack": [
        "FastAPI",
        "Passlib",
        "PyJWT"
      ]
    },
    {
      "name": "Dashboard Module",
      "description": "Protected user interface for viewing account data.",
      "tech_stack": [
        "FastAPI",
        "Jinja2",
        "HTML/CSS"
      ]
    },
    {
      "name": "Data Persistence Layer",
      "description": "Relational database management for user credentials and application state.",
      "tech_stack": [
        "PostgreSQL",
        "SQLAlchemy",
        "Alembic"
      ]
    }
  ],
  "constraints": {
    "security": [
      "Password hashing with bcrypt",
      "JWT token expiration",
      "Secure cookie handling"
    ],
    "compliance": [
      "None specified"
    ],
    "performance": [
      "Asyn